In [ ]:
import json
import pandas as pd
from pathlib import Path

In [ ]:
PROJECT_PATH = Path.cwd().parent
DATA_PATH = PROJECT_PATH / "data"
MAPPINGS_PATH = PROJECT_PATH / "mappings"
print(PROJECT_PATH)
print(DATA_PATH)
print(MAPPINGS_PATH)

In [ ]:
df = pd.read_csv(DATA_PATH / "revolut_dkk_2024-2026.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.isna().sum().to_dict()

In [ ]:
df[df.isna().any(axis=1)]

In [ ]:
for c in ["Type", "Product", "Currency", "State"]:
    print(df[c].unique(), end="\n\n\n")

In [ ]:
df[df["State"] == "REVERTED"]

In [ ]:
def clean_revolut_data(df):
    df = df[df["State"] != "REVERTED"]
    df = df.drop(columns=["Product", "Completed Date", "State"])
    df = df.rename(columns={
        "Type": "type",
        "Started Date": "date",
        "Description": "description",
        "Amount": "amount",
        "Fee": "fee",
        "Currency": "currency",
        "Balance": "balance"
    })
    df["date"] = pd.to_datetime(df["date"])
    print("Data frame is cleaned.")
    return df

df = clean_revolut_data(df)

In [ ]:
def dkk_to_euro_column(df):
    RATE = 7.46
    df["amount_euro"] = (df["amount"] / RATE).round(2)
    return df

df = dkk_to_euro_column(df)

In [ ]:
def extract_descrptions(df):
    
    print(f"{len(df['description'].unique())} unique descriptions detected.")
    
    descriptions = dict()
    for d in df["description"].unique():
        descriptions[d] = "blank"

    with open(MAPPINGS_PATH / "dkk_descriptions.json", "w") as f:
        json.dump(descriptions, f, sort_keys=True, ensure_ascii=False, indent=4)
    
extract_descrptions(df)       

In [ ]:
df[df["description"].str.contains("rannør", case=False, na=False)]

In [ ]:
def create_categories_column(df: pd.DataFrame):
    df = df.sort_values("date").reset_index(drop=True)
    file = MAPPINGS_PATH / "llm_dkk_categories.json"
    with open(file) as f:
        mappings = json.load(f)

    df["categories"] = df["description"].map(mappings)

    return df

df = create_categories_column(df)

In [ ]:
df

In [ ]:
def create_parquet(df: pd.DataFrame, parquet_name: str):
    out_dir = DATA_PATH / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_dir / parquet_name)

create_parquet(df, "revolut_dkk_2020_5-2026.parquet")

In [ ]:
df.columns

In [ ]:
Index(['date', 'description', 'amount', 'balance', 'source', 'category'], dtype='str')
Index(['type', 'date', 'description', 'amount', 'fee', 'currency', 'balance', 'categories'], dtype='str')
Index(['type', 'date', 'description', 'amount', 'fee', 'currency', 'balance', 'amount_euro', 'categories'], dtype='str')